### Bronze Layer


##StatsCanada
|

In [0]:
%sql
CREATE TABLE IF NOT EXISTS macro.bronze.statcan_raw (
  run_id STRING,
  ingested_at TIMESTAMP,
  request_key STRING,
  request_url STRING,
  vector_id BIGINT,
  http_status INT,
  payload STRING,
  payload_sha256 STRING
) USING DELTA;

CREATE TABLE IF NOT EXISTS macro.ops.ingestion_log (
  run_id STRING,
  source STRING,
  started_at TIMESTAMP,
  finished_at TIMESTAMP,
  status STRING,
  rows_fetched INT,
  rows_new INT,
  error_message STRING
) USING DELTA;

In [0]:
import requests, hashlib, uuid, time
from datetime import datetime, timezone

BASE = "https://www150.statcan.gc.ca/t1/wds/rest"
URL = f"{BASE}/getDataFromVectorsAndLatestNPeriods"
VECTORS = [2062815, 41690973]

def post_with_retry(url, payload, tries=3):
    last = None
    for attempt in range(1, tries + 1):
        try:
            r = requests.post(url, json=payload, timeout=30)
            if r.status_code == 200:
                return r
            last = f"HTTP {r.status_code}: {r.text[:200]}"
            if 400 <= r.status_code < 500 and r.status_code != 429:
                break                      # our mistake, retrying won't help
        except requests.RequestException as e:
            last = str(e)
        time.sleep(5 * attempt)            # wait longer each retry
    raise RuntimeError(last)

def ingest_statcan(latest_n=60):
    run_id = str(uuid.uuid4())
    started = datetime.now(timezone.utc)
    status, error, fetched = "SUCCESS", None, 0
    before = spark.table("macro.bronze.statcan_raw").count()
    try:
        rows = []
        for v in VECTORS:                  # one row per series, so each has its own hash
            r = post_with_retry(URL, [{"vectorId": v, "latestN": latest_n}])
            body = r.text
            rows.append((run_id, started, f"vector={v}|latestN={latest_n}", URL, v,
                         r.status_code, body,
                         hashlib.sha256(body.encode("utf-8")).hexdigest()))
        fetched = len(rows)
        schema = ("run_id string, ingested_at timestamp, request_key string, "
                  "request_url string, vector_id bigint, http_status int, "
                  "payload string, payload_sha256 string")
        spark.createDataFrame(rows, schema).createOrReplaceTempView("incoming")
        spark.sql("""
            MERGE INTO macro.bronze.statcan_raw t
            USING incoming s
            ON t.request_key = s.request_key AND t.payload_sha256 = s.payload_sha256
            WHEN NOT MATCHED THEN INSERT *""")
    except Exception as e:
        status, error = "FAILED", str(e)[:500]
    after = spark.table("macro.bronze.statcan_raw").count()
    log = [(run_id, "statcan", started, datetime.now(timezone.utc),
            status, fetched, after - before, error)]
    spark.createDataFrame(log, "run_id string, source string, started_at timestamp, "
                               "finished_at timestamp, status string, rows_fetched int, "
                               "rows_new int, error_message string") \
         .write.mode("append").saveAsTable("macro.ops.ingestion_log")
    print(status, "fetched:", fetched, "new:", after - before, error or "")

ingest_statcan()

SUCCESS fetched: 2 new: 2 


In [0]:
ingest_statcan()      # run it a second time
display(spark.table("macro.ops.ingestion_log").orderBy("started_at"))

SUCCESS fetched: 2 new: 0 


run_id,source,started_at,finished_at,status,rows_fetched,rows_new,error_message
6e960042-ebd2-4191-a98f-e0c69570bfce,statcan,2026-11-09T02:23:24.668Z,2026-11-09T02:23:36.913Z,SUCCESS,2,2,null
316cf6e5-5642-45de-b484-d48d2bf2748b,statcan,2026-11-09T02:24:12.363Z,2026-11-09T02:24:16.649Z,SUCCESS,2,0,null
3babf4bf-c7b3-40cd-be2d-ff7dee8ef22f,statcan,2026-11-09T02:24:45.036Z,2026-11-09T02:24:48.937Z,SUCCESS,2,0,null


## Bank Of canada Valet


In [0]:
%sql
CREATE TABLE IF NOT EXISTS macro.bronze.boc_raw (
  run_id STRING,
  ingested_at TIMESTAMP,
  request_key STRING,
  request_url STRING,
  series_code STRING,
  http_status INT,
  payload STRING,
  payload_sha256 STRING
) USING DELTA;

In [0]:
import requests, hashlib, uuid, time
from datetime import datetime, timezone

BOC_BASE = "https://www.bankofcanada.ca/valet/observations"
BOC_SERIES = ["FXUSDCAD", "V39079"]     # USD/CAD and the policy rate (verify V39079 below)

def get_with_retry(url, params, tries=3):
    last = None
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200:
                return r
            last = f"HTTP {r.status_code}: {r.text[:200]}"
            if 400 <= r.status_code < 500 and r.status_code != 429:
                break
        except requests.RequestException as e:
            last = str(e)
        time.sleep(5 * attempt)
    raise RuntimeError(last)

def ingest_boc(mode="incremental", start_date="2023-01-01"):
    run_id = str(uuid.uuid4())
    started = datetime.now(timezone.utc)
    status, error, fetched = "SUCCESS", None, 0
    before = spark.table("macro.bronze.boc_raw").count()
    try:
        rows = []
        for s in BOC_SERIES:
            url = f"{BOC_BASE}/{s}/json"
            if mode == "backfill":
                params, key = {"start_date": start_date}, f"series={s}|start={start_date}"
            else:
                params, key = {"recent": 30}, f"series={s}|recent=30"
            r = get_with_retry(url, params)
            body = r.text
            rows.append((run_id, started, key, r.url, s, r.status_code, body,
                         hashlib.sha256(body.encode("utf-8")).hexdigest()))
            time.sleep(1)                  # be polite to the API
        fetched = len(rows)
        schema = ("run_id string, ingested_at timestamp, request_key string, "
                  "request_url string, series_code string, http_status int, "
                  "payload string, payload_sha256 string")
        spark.createDataFrame(rows, schema).createOrReplaceTempView("incoming_boc")
        spark.sql("""
            MERGE INTO macro.bronze.boc_raw t
            USING incoming_boc s
            ON t.request_key = s.request_key AND t.payload_sha256 = s.payload_sha256
            WHEN NOT MATCHED THEN INSERT *""")
    except Exception as e:
        status, error = "FAILED", str(e)[:500]
    after = spark.table("macro.bronze.boc_raw").count()
    log = [(run_id, f"boc_{mode}", started, datetime.now(timezone.utc),
            status, fetched, after - before, error)]
    spark.createDataFrame(log, "run_id string, source string, started_at timestamp, "
                               "finished_at timestamp, status string, rows_fetched int, "
                               "rows_new int, error_message string") \
         .write.mode("append").saveAsTable("macro.ops.ingestion_log")
    print(status, "fetched:", fetched, "new:", after - before, error or "")

ingest_boc(mode="backfill")      # run once for history
ingest_boc(mode="incremental")   # then the daily-style run
ingest_boc(mode="incremental")   # idempotency test: should show new: 0

SUCCESS fetched: 2 new: 2 
SUCCESS fetched: 2 new: 2 
SUCCESS fetched: 2 new: 0 


Why Hash-Based is Preferred Here (The "Why")
Catching Silent Data Revisions: Macroeconomic data is heavily subject to retrospective revisions. Statistics Canada frequently revises historical unemployment or GDP numbers months after the initial release. If you rely on the hash, a revision changes the raw JSON text, generates a new hash, and forces the MERGE to capture the updated payload.

Safe Overlapping Windows: Your pipeline relies on fetching the recent=30 or latestN=36 periods every day. This creates massive data overlap by design. The hash perfectly handles this by ignoring the 29 days of data it has already seen and only writing a new row when the actual data payload increments or changes.

Bulletproof Recovery: If your Databricks cluster crashes midway through a run, or you decide to manually trigger a backfill for the last 6 months, the hash guarantees you will never duplicate a single record in your Bronze layer. It relies on the actual state of the data, not the state of the orchestrator.

Why Log-Based Idempotency Fails Here (The "Why Not")
Blind to Source Changes: Log-based idempotency only asks, "Did I run successfully today?" If Statistics Canada updates a faulty CPI number at 2:00 PM, and your log says you already ran successfully at 8:00 AM, a rerun will skip the API call. You will miss the critical data revision entirely.

Tightly Coupled to Execution, Not Data: Logs track the behavior of your Python script, whereas hashes track the actual reality of the data source. In a Bronze layer (which is meant to be a perfect historical ledger of source data), data reality is always more important than execution history.

State Desync Risks: If the pipeline inserts data into boc_raw but crashes right before appending the success metric to ingestion_log, the next run will see no success log and insert the data a second time, causing duplicates.

The Production Standard: A Hybrid Approach
In enterprise data engineering, you do not actually have to choose just one; you use them for different layers of the pipeline:

Use Logs for the Orchestrator (Airflow): Use logs or Airflow's built-in state tracking to ensure the DAG only triggers once per day, saving unnecessary compute costs and avoiding API rate limits.

Use Hashes for the Storage (Databricks/Delta): Keep the MERGE INTO ... payload_sha256 logic at the database level. If Airflow is accidentally triggered twice, or you force a manual rerun, the hash acts as the final bouncer at the door, ensuring your Delta tables remain pristine

In [0]:
import requests, json

r = requests.get("https://www.bankofcanada.ca/valet/series/V39079/json", timeout=30)
print(r.status_code)
data = r.json()
print(list(data.keys()))
print(json.dumps(data, indent=2)[:1200])

200
['terms', 'seriesDetails']
{
  "terms": {
    "url": "https://www.bankofcanada.ca/terms/"
  },
  "seriesDetails": {
    "name": "V39079",
    "label": "Target for the overnight rate (business daily)",
    "description": "Also called the policy interest rate, the average rate that the Bank of Canada wants to see in the market for overnight money market financing. (V39079)",
    "dateFormat": "daily",
    "metadata": {
      "frequency": "Business daily",
      "unit": "Percent",
      "unit_symbol": "%"
    }
  }
}


In [0]:
r = requests.get("https://www.bankofcanada.ca/valet/observations/V39079/json",
                 params={"recent": 3}, timeout=30)
print(r.status_code)
data = r.json()
print(list(data.keys()))
print(json.dumps(data["observations"], indent=2))
print(json.dumps(data["seriesDetail"], indent=2)[:600])

200
['terms', 'seriesDetail', 'observations']
[
  {
    "d": "2026-11-05",
    "V39079": {
      "v": "2.50"
    }
  },
  {
    "d": "2026-11-04",
    "V39079": {
      "v": "2.50"
    }
  },
  {
    "d": "2026-11-03",
    "V39079": {
      "v": "2.50"
    }
  }
]
{
  "V39079": {
    "label": "Target for the overnight rate (business daily)",
    "description": "Also called the policy interest rate, the average rate that the Bank of Canada wants to see in the market for overnight money market financing. (V39079)",
    "dimension": {
      "key": "d",
      "name": "Date"
    },
    "dateFormat": "daily",
    "metadata": {
      "unit": "Percent",
      "frequency": "Business daily"
    }
  }
}


In [0]:
%sql
SELECT 'boc_raw' AS tbl, COUNT(*) AS rows FROM macro.bronze.boc_raw
UNION ALL
SELECT 'statcan_raw', COUNT(*) FROM macro.bronze.statcan_raw;

tbl,rows
boc_raw,4
statcan_raw,2


In [0]:
%sql
SELECT series_code, request_key, http_status, length(payload) AS payload_chars,
       ingested_at
FROM macro.bronze.boc_raw
ORDER BY ingested_at;

series_code,request_key,http_status,payload_chars,ingested_at
FXUSDCAD,series=FXUSDCAD|start=2023-01-01,200,117693,2026-11-09T02:29:16.137Z
V39079,series=V39079|start=2023-01-01,200,118438,2026-11-09T02:29:16.137Z
FXUSDCAD,series=FXUSDCAD|recent=30,200,4233,2026-11-09T02:29:23.048Z
V39079,series=V39079|recent=30,200,4214,2026-11-09T02:29:23.048Z


In [0]:
%sql
SELECT
  series_code,
  o.d AS obs_date,
  o.v AS value_text
FROM (
  SELECT series_code,
         explode(from_json(get_json_object(payload, '$.observations'),
                 'array<struct<d:string,v:string>>')) AS o
  FROM macro.bronze.boc_raw
  WHERE request_key LIKE '%start=%'
)
LIMIT 200;


series_code,obs_date,value_text
FXUSDCAD,2023-01-03,null
FXUSDCAD,2023-01-04,null
FXUSDCAD,2023-01-05,null
FXUSDCAD,2023-01-06,null
FXUSDCAD,2023-01-09,null
FXUSDCAD,2023-01-10,null
FXUSDCAD,2023-01-11,null
FXUSDCAD,2023-01-12,null
FXUSDCAD,2023-01-13,null
FXUSDCAD,2023-01-16,null


In [0]:
%sql
SELECT series_code,
       get_json_object(payload, concat('$.observations[', i, '].d')) AS obs_date,
       get_json_object(payload, concat('$.observations[', i, '].', series_code, '.v')) AS value_text
FROM (
  SELECT series_code, payload,
         explode(sequence(0, size(from_json(get_json_object(payload, '$.observations'),
                                            'array<struct<d:string>>')) - 1)) AS i
  FROM macro.bronze.boc_raw
  WHERE request_key LIKE '%start=%'
)
ORDER BY series_code, obs_date
LIMIT 200;

series_code,obs_date,value_text
FXUSDCAD,2023-01-03,1.3658
FXUSDCAD,2023-01-04,1.3508
FXUSDCAD,2023-01-05,1.3564
FXUSDCAD,2023-01-06,1.3486
FXUSDCAD,2023-01-09,1.3376
FXUSDCAD,2023-01-10,1.3421
FXUSDCAD,2023-01-11,1.3425
FXUSDCAD,2023-01-12,1.3378
FXUSDCAD,2023-01-13,1.3406
FXUSDCAD,2023-01-16,1.3397


In [0]:
%sql
SELECT series_code,
       COUNT(*) AS rows,
       SUM(CASE WHEN value_text IS NULL THEN 1 ELSE 0 END) AS null_values,
       MIN(obs_date) AS first_date,
       MAX(obs_date) AS last_date
FROM (
  SELECT series_code,
         get_json_object(payload, concat('$.observations[', i, '].d')) AS obs_date,
         get_json_object(payload, concat('$.observations[', i, '].', series_code, '.v')) AS value_text
  FROM (
    SELECT series_code, payload,
           explode(sequence(0, size(from_json(get_json_object(payload, '$.observations'),
                                              'array<struct<d:string>>')) - 1)) AS i
    FROM macro.bronze.boc_raw
    WHERE request_key LIKE '%start=%'
  )
)
GROUP BY series_code;

series_code,rows,null_values,first_date,last_date
V39079,998,0,2023-01-03,2026-11-05
FXUSDCAD,960,0,2023-01-03,2026-11-06
